# 030 — A100 L pump: SYSON/SYSOFF serial test

Raw pyserial test of the RS-485 commands on **COM36**.

Protocol (manual `raw/Manuals/Pfeiffer_A100L_Manual.pdf`, ch. 13.3–13.6):
9600 baud, 8N1, no echo. Command `#<adr><ORDER><CR>` with adr `000`,
response `#000OK<CR>` or `#000ERRx<CR>`.

**Close the manufacturer program first** — it holds COM36.
Kernel: `esibd` env.

In [1]:
import serial

PORT = "COM36"
ADR = "000"

ser = serial.Serial(PORT, baudrate=9600, bytesize=8, parity="N", stopbits=1, timeout=2)

def cmd(order: str) -> bytes:
    ser.reset_input_buffer()
    msg = f"#{ADR}{order}\r".encode("ascii")
    ser.write(msg)
    resp = ser.read_until(b"\r")
    print(f"sent {msg!r} -> got {resp!r}")
    return resp

In [15]:
# read-only status query; decoded by parse_sta cell below
cmd("STA");

sent b'#000STA\r' -> got b'#000\x80\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'


In [16]:
# read-only status query first: char 5 of reply = 0 stopped / 1 running
cmd("STA");

sent b'#000STA\r' -> got b'#000\x80\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'


In [13]:
# read-only status query first: char 5 of reply = 0 stopped / 1 running
cmd("STA");

sent b'#000STA\r' -> got b'#000\x80\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'


In [14]:
# read-only status query first: char 5 of reply = 0 stopped / 1 running
cmd("STA");

sent b'#000STA\r' -> got b'#000\x80\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'


In [39]:
# decode STA reply — RS-485 layout, NO separator chars (32 payload bytes):
# A B 000 000 E 0000 0000 000 0000 00 abcdef ; status bytes are bit fields, bit 7 always set
def parse_sta(resp: bytes) -> dict:
    body = resp[4:-1]                      # strip '#000' + CR
    assert len(body) == 32, f"unexpected STA length {len(body)}: {resp!r}"
    A, E = body[0], body[8]
    a, b, c, d, e, f = body[26:32]
    return {
        "running":        bool(A & 0x40),           # STA A bit 6
        "mode":           "remote" if (E & 0x07) == 1 else "local",
        "temp_warning":   bool(e & 0x30),           # STA e motor-temp bit (4 or 5, manual table ambiguous)
        "temp_alarm":     bool(f & 0x30),           # STA f motor-temp bit
        "variator_alarm": bool(d & 0x01),
        "any_warning":    bool((a | c | e) & 0x7F),
        "any_alarm":      bool((b | d | f) & 0x7F),
    }

parse_sta(cmd("STA"))

sent b'#000STA\r' -> got b'#000\x80\x80000000\x8100000000000000000\x80\x80\x80\x80\x90\x80\r'


{'running': False,
 'mode': 'remote',
 'temp_warning': True,
 'temp_alarm': False,
 'variator_alarm': False,
 'any_warning': True,
 'any_alarm': False}

In [38]:
from time import sleep
while True:
    a = parse_sta(cmd("STA"))["temp_warning"]
    print(a)
    if a:
        cmd("SYSOFF")
        break
    sleep(1)

sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False
sent b'#000STA\r' -> got b'#000\xc0\x80000000\x8100000000000000000\x80\x80\x80\x80\x80\x80\r'
False


In [ ]:
cmd("SYSOFF");

In [37]:
# CHANGE A100: start pump
cmd("SYSON");  # expect b'#000OK\r'

sent b'#000SYSON\r' -> got b'#000 OK..\r'


In [28]:
# CHANGE A100: stop pump
cmd("SYSOFF");  # expect b'#000OK\r'

sent b'#000SYSOFF\r' -> got b'#000 OK..\r'


In [ ]:
ser.close()